In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

In [2]:
#Import our functions
from src.features.word2vec import (
    prepare_sentences,
    train_word2vec,
    create_avg_word2vec_vectors
)

In [3]:
#Load processed data
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_processed.csv"
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (120000, 9)
Test shape : (7600, 6)


In [4]:
#Prepare the Word2Vec training corpus
X_train_text = train_df["clean_text"].fillna("")
X_test_text = test_df["clean_text"].fillna("")

y_train = train_df["label"]
y_test = test_df["label"]

In [5]:
train_sentences = prepare_sentences(X_train_text)

print("Number of sentences:", len(train_sentences))
print("First sentence:")
print(train_sentences[0][:20])

Number of sentences: 120000
First sentence:
['wall', 'st', 'bear', 'claw', 'back', 'black', 'reuters', 'reuters', 'shortsellers', 'wall', 'street', 'dwindlingband', 'ultracynics', 'seeing', 'green']


In [6]:
#Train Word2Vec
word2vec_model = train_word2vec(
    sentences=train_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

In [7]:
#Check the vocabulary
print("Vocabulary size:", len(word2vec_model.wv))

Vocabulary size: 44734


In [8]:
#check the size
print(
    "Embedding dimension:",
    word2vec_model.vector_size
)

Embedding dimension: 100


In [9]:
#Look at a word vector
word = "computer"

if word in word2vec_model.wv:
    print(word2vec_model.wv[word])
else:
    print(f"'{word}' is not in the vocabulary.")

[ 4.62341279e-01  3.14971209e-01 -4.40010190e-01  1.58545718e-01
  3.06797266e-01  4.67654653e-02  6.37973785e-01  2.22957671e-01
  2.28608027e-01 -1.19286895e-01  2.75796145e-01 -1.24879457e-01
  1.94421366e-01  5.97903371e-01 -3.04799050e-01 -8.22026432e-02
  1.73765589e-02 -5.40067196e-01  2.72505164e-01 -8.14594269e-01
  4.83417273e-01  3.80164124e-02  1.22184478e-01  5.57382047e-01
 -3.10056610e-04  7.03358889e-01  6.87240422e-01 -2.04362437e-01
  3.22016209e-01  2.41299048e-02 -9.17520821e-02 -9.42224264e-02
 -1.95184782e-01  2.54814565e-01 -3.29394341e-02  2.27041915e-01
  3.28358948e-01  7.41669834e-01  1.40600786e-01  2.70151943e-02
  4.08823282e-01 -8.71458828e-01  4.07057106e-01 -1.72314793e-01
  7.43582726e-01  1.46679670e-01 -7.65664458e-01  3.04782033e-01
 -2.34210446e-01  6.52179241e-01  1.90426037e-01 -4.64494765e-01
 -2.26329565e-01  4.08938795e-01 -5.30420840e-01  4.23416466e-01
  1.88783407e-01 -4.91140872e-01  1.16215520e-01 -2.05519170e-01
 -4.79997516e-01  4.47585

In [10]:
#Check similar words
word = "computer"

if word in word2vec_model.wv:

    similar_words = word2vec_model.wv.most_similar(
        word,
        topn=10
    )

    for similar_word, similarity in similar_words:
        print(
            f"{similar_word:20s} {similarity:.4f}"
        )

pc                   0.6953
personal             0.6805
computersand         0.6796
machine              0.6597
noncampus            0.6537
personalcomputer     0.6492
surreptitiously      0.6413
bonusesfor           0.6412
varian               0.6378
outon                0.6355


In [11]:
#Convert articles to AvgWord2Vec vectors
X_train_avgw2v = create_avg_word2vec_vectors(
    word2vec_model,
    X_train_text
)

X_test_avgw2v = create_avg_word2vec_vectors(
    word2vec_model,
    X_test_text
)

In [13]:
import numpy as np

X_train_avgw2v = np.array(X_train_avgw2v)
X_test_avgw2v = np.array(X_test_avgw2v)

In [14]:
#Convert them to NumPy arrays:
print("Train shape:", X_train_avgw2v.shape)
print("Test shape :", X_test_avgw2v.shape)

Train shape: (120000, 100)
Test shape : (7600, 100)


In [15]:
#Train classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [16]:
models_w2v = {
    "AvgWord2Vec Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=1.0
    ),

    "AvgWord2Vec Linear SVM": LinearSVC(
        C=1.0
    )
}

In [17]:
#Train and evaluate:
from src.models.evaluate import evaluate_model

w2v_results = []
w2v_trained_models = {}

for model_name, model in models_w2v.items():

    print(f"\nTraining {model_name}...")

    model.fit(
        X_train_avgw2v,
        y_train
    )

    w2v_trained_models[model_name] = model

    result = evaluate_model(
        model,
        X_test_avgw2v,
        y_test,
        model_name
    )

    w2v_results.append(result)


Training AvgWord2Vec Logistic Regression...
AvgWord2Vec Logistic Regression
Accuracy : 0.8914
Precision: 0.8912
Recall   : 0.8914
F1 Score : 0.8913

Classification Report:
              precision    recall  f1-score   support

           1       0.90      0.89      0.89      1900
           2       0.95      0.97      0.96      1900
           3       0.85      0.85      0.85      1900
           4       0.86      0.86      0.86      1900

    accuracy                           0.89      7600
   macro avg       0.89      0.89      0.89      7600
weighted avg       0.89      0.89      0.89      7600


Training AvgWord2Vec Linear SVM...
AvgWord2Vec Linear SVM
Accuracy : 0.8934
Precision: 0.8932
Recall   : 0.8934
F1 Score : 0.8932

Classification Report:
              precision    recall  f1-score   support

           1       0.91      0.88      0.90      1900
           2       0.94      0.97      0.96      1900
           3       0.86      0.85      0.86      1900
           4       0

In [18]:
#Create the results table
w2v_results_df = pd.DataFrame(
    w2v_results
)

w2v_results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]
]

,model,accuracy,precision,recall,f1_score
0,AvgWord2Vec Logistic Regression,0.891447,0.891177,0.891447,0.891275
1,AvgWord2Vec Linear SVM,0.893421,0.893187,0.893421,0.893189


In [21]:
w2v_results_df = pd.DataFrame(w2v_results)

w2v_results_df[
    [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]
]

,model,accuracy,precision,recall,f1_score
0,AvgWord2Vec Logistic Regression,0.891447,0.891177,0.891447,0.891275
1,AvgWord2Vec Linear SVM,0.893421,0.893187,0.893421,0.893189


In [22]:
phase3_results = pd.read_csv(
    "../models/tfidf_results.csv"
)

phase3_results

,model,accuracy,precision,recall,f1_score
0,Naive Bayes,0.904211,0.903965,0.904211,0.903879
1,Logistic Regression,0.921053,0.920920,0.921053,0.920897
2,Linear SVM,0.923553,0.923494,0.923553,0.923457


In [23]:
#camparision with phase 3 results
comparison_df = pd.concat(
    [
        phase3_results,
        w2v_results_df
    ],
    ignore_index=True
)

comparison_df

,model,accuracy,precision,recall,f1_score,confusion_matrix
0,Naive Bayes,0.904211,0.903965,0.904211,0.903879,NaN
1,Logistic Regression,0.921053,0.920920,0.921053,0.920897,NaN
2,Linear SVM,0.923553,0.923494,0.923553,0.923457,NaN
3,AvgWord2Vec Logistic Regression,0.891447,0.891177,0.891447,0.891275,"[[1685, 64, 91, 60], [29, 1838, 16, 17], [78, ..."
4,AvgWord2Vec Linear SVM,0.893421,0.893187,0.893421,0.893189,"[[1676, 68, 95, 61], [23, 1849, 13, 15], [65, ..."
